# 3-10 分箱操作cut

分箱是把连续数值划分到若干区间，例如把考试成绩划分为不同等级。`pd.cut()` 按边界区间分箱。

> 约定：使用 `pd` 作为 Pandas 的别名，使用 `np` 作为 NumPy 的别名。本节可独立从上到下运行。

In [ ]:
import pandas as pd
import numpy as np

## 1. 准备成绩数据

固定随机种子，让每次运行得到相同的一组示例成绩。`randint(25, 100)` 包含 25，不包含 100。

In [ ]:
rng = np.random.RandomState(666)
score = rng.randint(25, 100, size=10)
score

## 2. cut 的常用参数

| 参数 | 含义 |
|---|---|
| `x` | 要分箱的一维数据 |
| `bins` | 箱数，或明确的边界列表 |
| `right` | 是否包含右端点，默认 `True` |
| `labels` | 给每个区间指定标签 |
| `include_lowest` | 是否包含第一个区间的最低边界 |

### 2.1 传入箱数

`bins=3` 表示按数据范围划分 3 个等宽区间。Pandas 会略微扩展范围以包含边缘数据，所以结果边界可能出现小数。

In [ ]:
pd.cut(score, bins=3)

### 2.2 指定区间边界

`bins=[0, 59, 70, 80, 100]` 划分出 4 个区间。默认 `right=True`：区间为 `(0, 59]`、`(59, 70]`、`(70, 80]`、`(80, 100]`。

In [ ]:
bins = [0, 59, 70, 80, 100]
pd.cut(score, bins=bins)

## 3. 左闭右开与左开右闭

`right=False` 改为 `[0, 59)`、`[59, 70)`、`[70, 80)`、`[80, 100)`。边界值的归属会改变。

In [ ]:
pd.cut(score, bins=3, right=False)

In [ ]:
pd.cut(score, bins=bins, right=False)

用恰好位于边界的数值比较更直观：

In [ ]:
boundary = pd.Series([0, 59, 70, 80, 100, 101])
pd.DataFrame({
    "数值": boundary,
    "右闭": pd.cut(boundary, bins=bins),
    "右闭且包含最低值": pd.cut(boundary, bins=bins, include_lowest=True),
    "左闭": pd.cut(boundary, bins=bins, right=False)
})

默认右闭时 0 不在区间中；设置 `include_lowest=True` 可以包含 0。左闭右开时 100 被排除，101 始终越界，越界结果为缺失值。最低区间可能显示稍低于 0 的边界，这是区间显示精度造成的表示方式。

## 4. 为区间设置标签

有 5 个边界就有 4 个区间，`labels` 必须提供 4 个标签。

In [ ]:
labels = ["low", "middle", "good", "perfect"]
pd.cut(score, bins=bins, labels=labels, include_lowest=True)

## 5. 将分箱结果加入表格并统计

分别保存区间和文字标签，便于核对每条数据。

In [ ]:
df = pd.DataFrame({"score": score})
df["range"] = pd.cut(df["score"], bins=bins, include_lowest=True)
df["label"] = pd.cut(df["score"], bins=bins, labels=labels, include_lowest=True)
df

In [ ]:
df.groupby("range", observed=False).agg(人数=("label", "count"))

分箱结果是分类数据。显式设置 `observed=False` 保留没有观测值的区间，便于展示完整等级；设置 `True` 则只显示有数据的分组。不同 Pandas 版本的默认值可能不同，因此这里明确写出。

## 6. 要点总结

- `cut` 按数值区间分箱，`bins` 可以是箱数，也可以是边界列表。
- 默认左开右闭，`right=False` 改为左闭右开。
- 边界值要检查；越界值会变为缺失值，`include_lowest` 可包含最低边界。
- 标签数量必须等于区间数量；分箱后可以用 `groupby` 统计。